# PlantGuard CNN Training — PlantVillage + MobileNetV2

This notebook trains a **real convolutional neural network** for plant disease detection and exports it for the PlantGuard web app.

**What happens, step by step:**
1. Check that a GPU is available (free Colab T4).
2. Install the needed Python packages.
3. Download the PlantVillage dataset from Kaggle (~54,000 leaf photos, 38 classes).
4. Train MobileNetV2 in two phases — first the new classification head, then fine-tuning the top backbone layers.
5. Evaluate on a held-out test set the model has never seen.
6. Export the model to TensorFlow.js (for the website) and TFLite (for mobile).
7. Download everything as one zip file.

> Expected result based on published work with this exact setup: **~92–96% test accuracy**. Anything in that range means the pipeline worked correctly.

## Step 1 — Check the GPU
Colab gives a free NVIDIA T4. Without it, training would take many hours.

In [ ]:
import tensorflow as tf
print("TensorFlow:", tf.__version__)
gpus = tf.config.list_physical_devices("GPU")
print("GPUs:", [g.name for g in gpus] or "NONE — enable GPU via Runtime > Change runtime type")
!nvidia-smi -L

## Step 2 — Install packages
`tensorflowjs` converts the trained model to the browser format used by the PlantGuard site.

In [ ]:
!pip install -q tensorflowjs scikit-learn kaggle matplotlib
print("packages installed")

## Step 3 — Kaggle credentials
1. Go to https://www.kaggle.com/settings → **API** → **Create New Token** (downloads `kaggle.json`).
2. Run the cell below and **upload `kaggle.json`** when asked.

In [ ]:
from google.colab import files
import os, shutil
os.makedirs(os.path.expanduser("~/.kaggle"), exist_ok=True)
uploaded = files.upload()  # <-- upload kaggle.json here
for name in uploaded:
    shutil.move(name, os.path.expanduser("~/.kaggle/kaggle.json"))
os.chmod(os.path.expanduser("~/.kaggle/kaggle.json"), 0o600)
print("Kaggle credentials saved.")

## Step 4 — Write the training scripts
These are the exact files from the PlantGuard repo (`cnn-training/`), recreated here so Colab runs byte-identical code.

In [ ]:
%%writefile data_utils.py
"""Shared dataset helpers for the PlantGuard CNN pipeline.

PlantVillage layout expected:
    <data_dir>/<class_name>/*.JPG            (38 class folders)

The abdallahalidev Kaggle copy nests this under "<root>/color/".
`discover_classes` handles both layouts.
"""
import json
import os
import random
from pathlib import Path

import numpy as np
import tensorflow as tf
from sklearn.model_selection import train_test_split
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input

IMG_EXTS = {".jpg", ".jpeg", ".png", ".JPG", ".JPEG", ".PNG"}


def resolve_class_root(data_dir: Path) -> Path:
    """Return the directory that directly contains the class folders."""
    data_dir = Path(data_dir)
    # Kaggle "plantvillage-dataset" copy: <root>/color/<38 classes>
    color = data_dir / "color"
    if color.is_dir() and _looks_like_class_root(color):
        return color
    if _looks_like_class_root(data_dir):
        return data_dir
    # One extra nesting level (e.g. "plantvillage dataset/color")
    for child in data_dir.iterdir():
        if child.is_dir():
            nested_color = child / "color"
            if nested_color.is_dir() and _looks_like_class_root(nested_color):
                return nested_color
            if _looks_like_class_root(child):
                return child
    raise FileNotFoundError(
        f"Could not find class folders under {data_dir}. "
        "Expected <data_dir>/<class>/*.jpg (or <data_dir>/color/<class>/*.jpg)."
    )


def _looks_like_class_root(d: Path) -> bool:
    min_dirs = int(os.environ.get("PLANTGUARD_MIN_CLASS_DIRS", "5"))
    subdirs = [c for c in d.iterdir() if c.is_dir()]
    if len(subdirs) < min_dirs:
        return False
    return any(_count_images(c) > 0 for c in subdirs)


def _count_images(d: Path) -> int:
    return sum(1 for f in d.iterdir() if f.is_file() and f.suffix in IMG_EXTS)


def discover_classes(data_dir: Path):
    """Return (class_root, sorted class names)."""
    root = resolve_class_root(Path(data_dir))
    classes = sorted([c.name for c in root.iterdir() if c.is_dir() and _count_images(c) > 0])
    if not classes:
        raise FileNotFoundError(f"No class folders with images found under {root}")
    return root, classes


def build_file_index(class_root: Path, classes):
    """Return list of (filepath, label_index)."""
    index = []
    for i, cls in enumerate(classes):
        for f in sorted((class_root / cls).iterdir()):
            if f.is_file() and f.suffix in IMG_EXTS:
                index.append((str(f), i))
    return index


def stratified_split(file_index, seed=42, train_ratio=0.8, val_ratio=0.1):
    """Stratified 80/10/10 split. Returns dict with train/val/test lists of (path, label)."""
    paths = [p for p, _ in file_index]
    labels = [l for _, l in file_index]
    p_train, p_rest, y_train, y_rest = train_test_split(
        paths, labels, train_size=train_ratio, random_state=seed, stratify=labels
    )
    val_size = val_ratio / (1.0 - train_ratio)
    p_val, p_test, y_val, y_test = train_test_split(
        p_rest, y_rest, train_size=val_size, random_state=seed, stratify=y_rest
    )
    return {
        "train": list(zip(p_train, y_train)),
        "val": list(zip(p_val, y_val)),
        "test": list(zip(p_test, y_test)),
    }


def save_split_manifest(path: Path, splits):
    serial = {k: [{"path": p, "label": int(l)} for p, l in v] for k, v in splits.items()}
    Path(path).write_text(json.dumps(serial, indent=2))


def load_split_manifest(path: Path):
    raw = json.loads(Path(path).read_text())
    return {k: [(e["path"], e["label"]) for e in v] for k, v in raw.items()}


def _decode(path, label, img_size):
    img = tf.io.read_file(path)
    # PlantVillage ships JPEG; tolerate PNG as well (extension-based branch).
    img = tf.cond(
        tf.strings.regex_full_match(path, ".*\\.[pP][nN][gG]$"),
        lambda: tf.image.decode_png(img, channels=3),
        lambda: tf.image.decode_jpeg(img, channels=3),
    )
    img = tf.image.resize(img, [img_size, img_size])
    img = preprocess_input(tf.cast(img, tf.float32))  # scale to [-1, 1] for MobileNetV2
    return img, label


def make_dataset(pairs, img_size, batch_size, num_classes, augment=False, shuffle=False, seed=42):
    paths = [p for p, _ in pairs]
    labels = [l for _, l in pairs]
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    if shuffle:
        ds = ds.shuffle(buffer_size=min(len(paths), 8192), seed=seed, reshuffle_each_iteration=True)
    ds = ds.map(lambda p, l: _decode(p, l, img_size), num_parallel_calls=tf.data.AUTOTUNE)
    if augment:
        aug = tf.keras.Sequential(
            [
                tf.keras.layers.RandomFlip("horizontal"),
                tf.keras.layers.RandomRotation(0.2),
                tf.keras.layers.RandomZoom(0.2),
                tf.keras.layers.RandomContrast(0.2),
            ]
        )
        ds = ds.map(lambda x, y: (aug(x, training=True), y), num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.map(
        lambda x, y: (x, tf.one_hot(y, num_classes)), num_parallel_calls=tf.data.AUTOTUNE
    )
    ds = ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)
    return ds


def make_eval_dataset(pairs, img_size, batch_size):
    """No augmentation / no one-hot: returns (images, int labels) for sklearn metrics."""
    paths = [p for p, _ in pairs]
    labels = np.array([l for _, l in pairs], dtype=np.int32)
    ds = tf.data.Dataset.from_tensor_slices(paths)
    ds = ds.map(
        lambda p: _decode(p, 0, img_size)[0], num_parallel_calls=tf.data.AUTOTUNE
    )
    ds = ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)
    return ds, labels


def class_distribution(pairs, classes):
    counts = {c: 0 for c in classes}
    for _, l in pairs:
        counts[classes[l]] += 1
    return counts


def compute_class_weights(train_pairs, num_classes):
    """Inverse-frequency weights to counter PlantVillage class imbalance."""
    labels = np.array([l for _, l in train_pairs])
    total = len(labels)
    weights = {}
    for c in range(num_classes):
        n = int((labels == c).sum())
        weights[c] = total / (num_classes * n) if n else 1.0
    return weights


def set_seeds(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)


In [ ]:
%%writefile download_data.py
#!/usr/bin/env python3
"""Download the PlantVillage dataset from Kaggle and verify it.

Usage:
    python download_data.py --out_dir ./data                       # download + extract + verify
    python download_data.py --out_dir ./data --verify-only          # only verify an existing dir
    python download_data.py --out_dir ./data --dataset-slug other/slug

Needs a Kaggle API token (~/.kaggle/kaggle.json). On Colab, upload kaggle.json
via files.upload() first — the notebook does this for you.

The Kaggle copy "abdallahalidev/plantvillage-dataset" (~54k images, 38 classes)
ships three variants: color/, grayscale/, segmented/. We use the COLOR variant.
"""
import argparse
import json
import shutil
import subprocess
import sys
import zipfile
from pathlib import Path

from data_utils import discover_classes, build_file_index, class_distribution

DEFAULT_SLUG = "abdallahalidev/plantvillage-dataset"
EXPECTED_CLASSES = 38
MIN_IMAGES_PER_CLASS = 100  # sanity floor; real PlantVillage classes are much larger


def run(cmd):
    print("+", " ".join(cmd), flush=True)
    subprocess.run(cmd, check=True)


def ensure_kaggle_auth():
    kaggle_json = Path.home() / ".kaggle" / "kaggle.json"
    if not kaggle_json.exists():
        sys.exit(
            "ERROR: Kaggle credentials not found at ~/.kaggle/kaggle.json\n"
            "  1. Go to https://www.kaggle.com/settings -> API -> Create New Token\n"
            "  2. Save it as ~/.kaggle/kaggle.json and chmod 600 it.\n"
            "  (On Colab the notebook uploads it for you.)"
        )


def download(slug: str, dest: Path):
    ensure_kaggle_auth()
    dest.mkdir(parents=True, exist_ok=True)
    run(["kaggle", "datasets", "download", "-d", slug, "-p", str(dest)])
    zips = list(dest.glob("*.zip"))
    if not zips:
        sys.exit(f"ERROR: no .zip downloaded into {dest}")
    for z in zips:
        print(f"Extracting {z.name} ...", flush=True)
        with zipfile.ZipFile(z, "r") as zf:
            zf.extractall(dest)
        z.unlink()


def verify(data_dir: Path, out_dir: Path):
    class_root, classes = discover_classes(data_dir)
    print(f"Class root : {class_root}")
    print(f"Classes    : {len(classes)}")
    index = build_file_index(class_root, classes)
    dist = class_distribution(index, classes)

    print(f"\n{'class':45s} {'images':>8s}")
    print("-" * 56)
    for cls in classes:
        print(f"{cls:45s} {dist[cls]:>8d}")
    total = sum(dist.values())
    print("-" * 56)
    print(f"{'TOTAL':45s} {total:>8d}")

    problems = []
    if len(classes) != EXPECTED_CLASSES:
        problems.append(f"expected {EXPECTED_CLASSES} classes, found {len(classes)}")
    small = [c for c in classes if dist[c] < MIN_IMAGES_PER_CLASS]
    if small:
        problems.append(f"classes with < {MIN_IMAGES_PER_CLASS} images: {small}")

    manifest = {
        "dataset": "PlantVillage (color variant)",
        "class_root": str(class_root),
        "num_classes": len(classes),
        "total_images": total,
        "classes": classes,
        "class_counts": dist,
    }
    manifest_path = out_dir / "data_manifest.json"
    manifest_path.write_text(json.dumps(manifest, indent=2))
    print(f"\nManifest written to {manifest_path}")

    if problems:
        print("\nVERIFICATION PROBLEMS:")
        for p in problems:
            print("  -", p)
        sys.exit(1)
    print("\nVerification OK: 38 classes, all counts healthy.")


def main():
    ap = argparse.ArgumentParser(description="Download + verify the PlantVillage dataset.")
    ap.add_argument("--out_dir", required=True, help="Directory for data + manifest.")
    ap.add_argument("--dataset-slug", default=DEFAULT_SLUG)
    ap.add_argument("--verify-only", action="store_true",
                    help="Skip download; only verify --data-dir.")
    ap.add_argument("--data-dir", default=None,
                    help="Dataset location (default: <out_dir>/plantvillage).")
    args = ap.parse_args()

    out_dir = Path(args.out_dir)
    data_dir = Path(args.data_dir) if args.data_dir else out_dir / "plantvillage"

    if not args.verify_only:
        # keep any previous download out of the way, then fetch fresh
        if data_dir.exists():
            print(f"Removing previous download at {data_dir}")
            shutil.rmtree(data_dir)
        download(args.dataset_slug, data_dir)

    verify(data_dir, out_dir)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile train.py
#!/usr/bin/env python3
"""Train the PlantGuard CNN: MobileNetV2 (ImageNet) + custom head, 38 PlantVillage classes.

Two-phase transfer learning:
    Phase 1: backbone frozen, train only the classification head.
    Phase 2: unfreeze the top N backbone layers (BatchNorm kept frozen),
             fine-tune with a small learning rate.

Usage:
    python train.py --data_dir ./data/plantvillage --out_dir ./outputs
    python train.py --data_dir ./data/plantvillage --epochs1 2 --epochs2 1 --batch_size 8  # smoke test
"""
import argparse
import json
import time
from pathlib import Path

import tensorflow as tf

from data_utils import (
    discover_classes,
    build_file_index,
    stratified_split,
    save_split_manifest,
    make_dataset,
    compute_class_weights,
    set_seeds,
)


def build_model(img_size, num_classes, dropout=0.2):
    base = tf.keras.applications.MobileNetV2(
        input_shape=(img_size, img_size, 3),
        include_top=False,
        weights="imagenet",
    )
    base.trainable = False  # phase 1: frozen
    inputs = tf.keras.Input(shape=(img_size, img_size, 3))
    x = base(inputs, training=False)
    x = tf.keras.layers.GlobalAveragePooling2D()(x)
    x = tf.keras.layers.Dropout(dropout)(x)
    outputs = tf.keras.layers.Dense(num_classes, activation="softmax")(x)
    model = tf.keras.Model(inputs, outputs, name="plantguard_mobilenetv2")
    return model, base


def main():
    ap = argparse.ArgumentParser(description="Train PlantGuard CNN on PlantVillage.")
    ap.add_argument("--data_dir", required=True)
    ap.add_argument("--out_dir", default="./outputs")
    ap.add_argument("--img_size", type=int, default=224, help="MobileNetV2 native input size.")
    ap.add_argument("--batch_size", type=int, default=32)
    ap.add_argument("--epochs1", type=int, default=10, help="Phase-1 (frozen backbone) epochs.")
    ap.add_argument("--epochs2", type=int, default=20, help="Phase-2 (fine-tune) epochs.")
    ap.add_argument("--lr1", type=float, default=1e-3)
    ap.add_argument("--lr2", type=float, default=1e-5)
    ap.add_argument("--fine_tune_layers", type=int, default=30,
                    help="How many top backbone layers to unfreeze in phase 2.")
    ap.add_argument("--dropout", type=float, default=0.2)
    ap.add_argument("--seed", type=int, default=42)
    ap.add_argument("--mixed_precision", action="store_true",
                    help="Enable fp16 mixed precision (recommended on Colab T4).")
    args = ap.parse_args()

    set_seeds(args.seed)
    if args.mixed_precision:
        tf.keras.mixed_precision.set_global_policy("mixed_float16")
        print("Mixed precision enabled (fp16).")

    gpus = tf.config.list_physical_devices("GPU")
    print(f"GPUs visible: {len(gpus)}")

    out_dir = Path(args.out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    # ---- data ----
    class_root, classes = discover_classes(args.data_dir)
    num_classes = len(classes)
    print(f"Classes: {num_classes}  ({classes[0]} ... {classes[-1]})")
    index = build_file_index(class_root, classes)
    print(f"Total images: {len(index)}")
    splits = stratified_split(index, seed=args.seed)
    for k, v in splits.items():
        print(f"  {k:6s}: {len(v)} images")
    save_split_manifest(out_dir / "split_manifest.json", splits)
    (out_dir / "labels.json").write_text(json.dumps(classes, indent=2))

    train_ds = make_dataset(splits["train"], args.img_size, args.batch_size, num_classes,
                            augment=True, shuffle=True, seed=args.seed)
    val_ds = make_dataset(splits["val"], args.img_size, args.batch_size, num_classes)
    class_weights = compute_class_weights(splits["train"], num_classes)

    # ---- model ----
    model, base = build_model(args.img_size, num_classes, args.dropout)
    model.summary(print_fn=print)

    ckpt = out_dir / "best_model.keras"
    callbacks = [
        tf.keras.callbacks.ModelCheckpoint(str(ckpt), monitor="val_accuracy",
                                           save_best_only=True, mode="max", verbose=1),
        tf.keras.callbacks.EarlyStopping(monitor="val_accuracy", patience=4,
                                         restore_best_weights=True, verbose=1),
        tf.keras.callbacks.CSVLogger(str(out_dir / "training_log.csv")),
    ]

    history_all = {}

    # ---- phase 1: frozen backbone ----
    print("\n===== PHASE 1: training head (backbone frozen) =====")
    model.compile(optimizer=tf.keras.optimizers.Adam(args.lr1),
                  loss="categorical_crossentropy", metrics=["accuracy"])
    t0 = time.time()
    h1 = model.fit(train_ds, validation_data=val_ds, epochs=args.epochs1,
                   class_weight=class_weights, callbacks=callbacks)
    print(f"Phase 1 took {(time.time() - t0) / 60:.1f} min")
    history_all["phase1"] = {k: [float(v) for v in vals] for k, vals in h1.history.items()}

    # ---- phase 2: fine-tune top layers ----
    print(f"\n===== PHASE 2: fine-tuning top {args.fine_tune_layers} backbone layers =====")
    base.trainable = True
    for layer in base.layers[: -args.fine_tune_layers]:
        layer.trainable = False
    # Keep BatchNorm frozen: with small batches its statistics get corrupted.
    for layer in base.layers[-args.fine_tune_layers:]:
        if isinstance(layer, tf.keras.layers.BatchNormalization):
            layer.trainable = False
    trainable = sum(1 for l in base.layers if l.trainable)
    print(f"Backbone layers trainable in phase 2: {trainable}/{len(base.layers)}")

    model.compile(optimizer=tf.keras.optimizers.Adam(args.lr2),
                  loss="categorical_crossentropy", metrics=["accuracy"])
    callbacks2 = [
        tf.keras.callbacks.ModelCheckpoint(str(ckpt), monitor="val_accuracy",
                                           save_best_only=True, mode="max", verbose=1),
        tf.keras.callbacks.EarlyStopping(monitor="val_accuracy", patience=5,
                                         restore_best_weights=True, verbose=1),
        tf.keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5,
                                             patience=2, verbose=1),
        tf.keras.callbacks.CSVLogger(str(out_dir / "training_log.csv"), append=True),
    ]
    t0 = time.time()
    h2 = model.fit(train_ds, validation_data=val_ds, epochs=args.epochs2,
                   class_weight=class_weights, callbacks=callbacks2)
    print(f"Phase 2 took {(time.time() - t0) / 60:.1f} min")
    history_all["phase2"] = {k: [float(v) for v in vals] for k, vals in h2.history.items()}

    # ---- save ----
    (out_dir / "history.json").write_text(json.dumps(history_all, indent=2))
    best_val = max(h1.history["val_accuracy"] + h2.history["val_accuracy"])
    print(f"\nBest val_accuracy across both phases: {best_val:.4f}")
    print(f"Best checkpoint: {ckpt}")
    print(f"Labels: {out_dir / 'labels.json'}")
    print(f"Split manifest: {out_dir / 'split_manifest.json'}")
    print("\nNext: python evaluate.py --model outputs/best_model.keras --out_dir outputs")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile evaluate.py
#!/usr/bin/env python3
"""Evaluate a trained PlantGuard CNN on the held-out TEST split.

Reads the exact test file list from split_manifest.json (written by train.py),
so the test set is identical to training time — no re-splitting, no leakage.

Reports: test accuracy, top-3 accuracy, per-class precision/recall/F1,
macro/weighted averages, and the top-5 most-confused class pairs.
Writes everything to metrics.json.

Usage:
    python evaluate.py --model ./outputs/best_model.keras --out_dir ./outputs
"""
import argparse
import json
from pathlib import Path

import numpy as np
import tensorflow as tf
from sklearn.metrics import (accuracy_score, classification_report,
                             confusion_matrix)

from data_utils import load_split_manifest, make_eval_dataset, set_seeds


def top_confused_pairs(cm, classes, k=5):
    pairs = []
    n = cm.shape[0]
    for i in range(n):
        for j in range(n):
            if i != j and cm[i, j] > 0:
                pairs.append({"true": classes[i], "predicted": classes[j],
                              "count": int(cm[i, j])})
    pairs.sort(key=lambda d: d["count"], reverse=True)
    return pairs[:k]


def main():
    ap = argparse.ArgumentParser(description="Evaluate PlantGuard CNN on the test split.")
    ap.add_argument("--model", required=True, help="Path to trained .keras model.")
    ap.add_argument("--out_dir", required=True,
                    help="Dir with split_manifest.json + labels.json (from train.py).")
    ap.add_argument("--batch_size", type=int, default=64)
    ap.add_argument("--img_size", type=int, default=224)
    ap.add_argument("--seed", type=int, default=42)
    ap.add_argument("--out", default=None, help="metrics.json path (default: <out_dir>/metrics.json).")
    args = ap.parse_args()

    set_seeds(args.seed)
    out_dir = Path(args.out_dir)
    classes = json.loads((out_dir / "labels.json").read_text())
    splits = load_split_manifest(out_dir / "split_manifest.json")
    test_pairs = splits["test"]
    print(f"Test images: {len(test_pairs)}, classes: {len(classes)}")

    model = tf.keras.models.load_model(args.model)
    test_ds, y_true = make_eval_dataset(test_pairs, args.img_size, args.batch_size)

    print("Running inference on test set ...")
    y_prob = model.predict(test_ds, verbose=1)
    y_pred = np.argmax(y_prob, axis=1)

    acc = accuracy_score(y_true, y_pred)
    # Manual top-k (avoids sklearn's binary-vs-multiclass type dispatch on tiny splits).
    k = min(3, len(classes))
    topk_idx = np.argsort(y_prob, axis=1)[:, -k:]
    top3 = float(np.mean([y_true[i] in topk_idx[i] for i in range(len(y_true))]))
    report = classification_report(y_true, y_pred, labels=list(range(len(classes))),
                                   target_names=classes,
                                   output_dict=True, zero_division=0)
    cm = confusion_matrix(y_true, y_pred, labels=list(range(len(classes))))

    per_class = {
        cls: {"precision": round(report[cls]["precision"], 4),
              "recall": round(report[cls]["recall"], 4),
              "f1": round(report[cls]["f1-score"], 4),
              "support": int(report[cls]["support"])}
        for cls in classes
    }
    worst5 = sorted(per_class.items(), key=lambda kv: kv[1]["f1"])[:5]

    metrics = {
        "model": str(args.model),
        "test_images": len(test_pairs),
        "num_classes": len(classes),
        "test_accuracy": round(float(acc), 4),
        "test_top3_accuracy": round(float(top3), 4),
        "macro_avg": {k: round(float(report["macro avg"][k]), 4)
                      for k in ("precision", "recall", "f1-score")},
        "weighted_avg": {k: round(float(report["weighted avg"][k]), 4)
                         for k in ("precision", "recall", "f1-score")},
        "per_class": per_class,
        "worst_5_classes_by_f1": [{"class": c, **m} for c, m in worst5],
        "top_5_confused_pairs": top_confused_pairs(cm, classes),
    }

    out_path = Path(args.out) if args.out else out_dir / "metrics.json"
    out_path.write_text(json.dumps(metrics, indent=2))

    print(f"\nTest accuracy     : {acc:.4f}")
    print(f"Test top-3 acc    : {top3:.4f}")
    print(f"Macro F1          : {report['macro avg']['f1-score']:.4f}")
    print(f"Weighted F1       : {report['weighted avg']['f1-score']:.4f}")
    print("\nWorst 5 classes by F1:")
    for c, m in worst5:
        print(f"  {c:45s} F1={m['f1']:.3f} (n={m['support']})")
    print("\nTop 5 confused pairs (true -> predicted):")
    for p in metrics["top_5_confused_pairs"]:
        print(f"  {p['true']} -> {p['predicted']}  ({p['count']} images)")
    print(f"\nFull metrics written to {out_path}")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile export.py
#!/usr/bin/env python3
"""Export a trained PlantGuard CNN for deployment.

Produces:
    <out_dir>/tfjs/            TensorFlow.js model (model.json + weight shards)
                               -> loaded in the browser by the PlantGuard web app
    <out_dir>/plantguard_cnn.tflite
                               -> TFLite (dynamic-range quantized), for future mobile use
    <out_dir>/labels.json      copy of the 38 class names (index-aligned)
    <out_dir>/export_manifest.json   sizes + input spec for the integration

Input spec (both formats): 224x224 RGB, float32, MobileNetV2 preprocessing
(i.e. pixel values scaled to [-1, 1]:  x/127.5 - 1).

Usage:
    python export.py --model ./outputs/best_model.keras --out_dir ./outputs/export
"""
import argparse
import json
import shutil
from pathlib import Path

import numpy as np
import tensorflow as tf


def _import_tfjs():
    """Import tensorflowjs, tolerating a broken tensorflow-decision-forests
    binary (it is eagerly imported by tensorflowjs but only used for the
    TF-DF model-conversion path, which we never exercise for Keras models).
    Where the real package imports fine (e.g. Colab) this is a no-op."""
    import sys
    import types

    try:
        import tensorflow_decision_forests  # noqa: F401
    except Exception:
        sys.modules.setdefault(
            "tensorflow_decision_forests",
            types.ModuleType("tensorflow_decision_forests"),
        )
    import tensorflowjs as tfjs

    return tfjs


def export_tfjs(model, tfjs_dir: Path):
    tfjs = _import_tfjs()
    if tfjs_dir.exists():
        shutil.rmtree(tfjs_dir)
    tfjs_dir.mkdir(parents=True)
    tfjs.converters.save_keras_model(model, str(tfjs_dir))
    files = sorted(tfjs_dir.glob("*"))
    total = sum(f.stat().st_size for f in files if f.is_file())
    return {"dir": str(tfjs_dir), "files": [f.name for f in files],
            "total_bytes": total}


def export_tflite(model, out_path: Path):
    converter = tf.lite.TFLiteConverter.from_keras_model(model)
    converter.optimizations = [tf.lite.Optimize.DEFAULT]  # dynamic-range quantization
    tflite_model = converter.convert()
    out_path.write_bytes(tflite_model)
    return {"path": str(out_path), "bytes": len(tflite_model)}


def main():
    ap = argparse.ArgumentParser(description="Export PlantGuard CNN to TF.js + TFLite.")
    ap.add_argument("--model", required=True, help="Trained .keras model.")
    ap.add_argument("--out_dir", required=True)
    ap.add_argument("--labels", default=None,
                    help="labels.json (default: alongside --model).")
    args = ap.parse_args()

    out_dir = Path(args.out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    labels_src = Path(args.labels) if args.labels else Path(args.model).parent / "labels.json"

    print(f"Loading {args.model} ...")
    model = tf.keras.models.load_model(args.model)
    print(f"Input: {model.input_shape}, output: {model.output_shape}")

    print("Exporting TensorFlow.js ...")
    tfjs_info = export_tfjs(model, out_dir / "tfjs")
    print(f"  {tfjs_info['total_bytes'] / 1e6:.2f} MB in {len(tfjs_info['files'])} files")

    print("Exporting TFLite (dynamic-range quantized) ...")
    tflite_info = export_tflite(model, out_dir / "plantguard_cnn.tflite")
    print(f"  {tflite_info['bytes'] / 1e6:.2f} MB")

    shutil.copy(labels_src, out_dir / "labels.json")

    # Sanity check: run one dummy inference through the TFLite model.
    interpreter = tf.lite.Interpreter(model_path=str(out_dir / "plantguard_cnn.tflite"))
    interpreter.allocate_tensors()
    inp = interpreter.get_input_details()[0]
    dummy = np.zeros(inp["shape"], dtype=np.float32)
    interpreter.set_tensor(inp["index"], dummy)
    interpreter.invoke()
    out = interpreter.get_output_details()[0]
    probs = interpreter.get_tensor(out["index"])[0]
    assert probs.shape[0] == model.output_shape[-1]
    print(f"TFLite sanity check OK: output sums to {probs.sum():.3f}")

    manifest = {
        "source_model": str(args.model),
        "input": {"height": 224, "width": 224, "channels": 3, "dtype": "float32",
                  "preprocessing": "x = pixel/127.5 - 1  (MobileNetV2, range [-1,1])"},
        "num_classes": model.output_shape[-1],
        "tfjs": tfjs_info,
        "tflite": tflite_info,
        "labels": str(out_dir / "labels.json"),
    }
    (out_dir / "export_manifest.json").write_text(json.dumps(manifest, indent=2))
    print(f"\nManifest: {out_dir / 'export_manifest.json'}")
    print("Tip: for a ~2x smaller browser model, re-run the TF.js export with float16 "
          "quantization:\n"
          "  tensorflowjs_converter --input_format=keras --quantize_float16 \\\n"
          "      outputs/best_model.keras outputs/export/tfjs_fp16")


if __name__ == "__main__":
    main()


## Step 5 — Download + verify PlantVillage
Downloads ~800MB–1GB. The color variant is used (38 classes).

In [ ]:
!python download_data.py --out_dir /content/data

## Step 6 — Train (the long step, ~1–2 hours on a T4)

**Phase 1** (frozen backbone): only the new classification head learns — fast and stable.
**Phase 2** (fine-tuning): the top 30 backbone layers unfreeze with a tiny learning rate, adapting ImageNet features to leaf-disease patterns. BatchNorm layers stay frozen.

Class weights counter the dataset imbalance (some diseases have far fewer photos). The best checkpoint by validation accuracy is saved automatically.

In [ ]:
!python train.py --data_dir /content/data/plantvillage --out_dir /content/outputs \
    --epochs1 10 --epochs2 25 --batch_size 32 --mixed_precision

## Step 7 — Evaluate on the held-out test set
The test split was fixed before training and never touched until now — this number is the honest accuracy.

In [ ]:
!python evaluate.py --model /content/outputs/best_model.keras --out_dir /content/outputs

## Step 8 — Export for deployment
- **TensorFlow.js** → runs in the browser; becomes PlantGuard's primary diagnosis engine.
- **TFLite** (quantized) → for a future mobile app.

In [ ]:
!python export.py --model /content/outputs/best_model.keras --out_dir /content/outputs/export

## Step 9 — Download everything
One zip with the model, browser files, labels, metrics, and training logs. This is what gets wired into the PlantGuard website.

In [ ]:
!cd /content/outputs && zip -qr /content/plantguard_cnn_artifacts.zip . -x ".*" 
from google.colab import files
files.download("/content/plantguard_cnn_artifacts.zip")
print("Done. Next: follow INTEGRATION_PLAN.md in the repo to wire the model into the app.")